# 03 Test

This notebook runs the same prompt set against both direct Azure OpenAI deployments and the APIM gateway so you can compare responses side by side.

## Load Environment

In [7]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', '')
AZURE_OPENAI_SECONDARY_DEPLOYMENT = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', '')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_SECONDARY_PATH = os.getenv('APIM_SECONDARY_CHAT_COMPLETIONS_PATH', '/openai51/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

print(f'Azure OpenAI endpoint: {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment: {AZURE_OPENAI_DEPLOYMENT}')
print(f'Secondary deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
print(f'APIM endpoint: {APIM_ENDPOINT}')
print(f'APIM primary path: {APIM_PATH}')
print(f'APIM secondary path: {APIM_SECONDARY_PATH}')

Azure OpenAI endpoint: https://aoaievalgw021uks.openai.azure.com
Primary deployment: chat4o
Secondary deployment: chat51
APIM endpoint: https://apim-evalgw02-1-uks.azure-api.net
APIM primary path: /openai/chat/completions
APIM secondary path: /openai51/chat/completions


## Test Each Target

In [8]:
import requests

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is not set in env/.env')

test_prompts = [
    'What is the capital of France?',
    'Explain machine learning in one sentence.',
    'Name three primary colors.'
]

results = []

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== Primary Deployment Prompt {prompt_index}: {prompt} =====')

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'direct-primary',
        'label': f'direct:{AZURE_OPENAI_DEPLOYMENT}',
        'deployment': AZURE_OPENAI_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')


===== Primary Deployment Prompt 1: What is the capital of France? =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
The capital of France is Paris.

===== Primary Deployment Prompt 2: Explain machine learning in one sentence. =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed.

===== Primary Deployment Prompt 3: Name three primary colors. =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
The three primary colors are **red**, **blue**, and **yellow**.


## Test Direct Secondary Deployment

In [9]:
import requests

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is not set in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== Secondary Deployment Prompt {prompt_index}: {prompt} =====')

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_SECONDARY_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'direct-secondary',
        'label': f'direct:{AZURE_OPENAI_SECONDARY_DEPLOYMENT}',
        'deployment': AZURE_OPENAI_SECONDARY_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')


===== Secondary Deployment Prompt 1: What is the capital of France? =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.",
    "type": "invalid_request_error",
    "param": "max_tokens",
    "code": "unsupported_parameter"
  }
}

===== Secondary Deployment Prompt 2: Explain machine learning in one sentence. =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.",
    "type": "invalid_request_error",
    "param": "max_tokens",
    "code": "unsupported_parameter"
  }
}

===== Secondary Deployment Prompt 3: Name three primary colors. =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not s

## Test APIM Gateway Primary

In [10]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== APIM Primary Prompt {prompt_index}: {prompt} =====')

    url = f"{APIM_ENDPOINT}{APIM_PATH}"
    headers = {
        'Ocp-Apim-Subscription-Key': APIM_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'apim-primary',
        'label': 'apim:gateway-to-primary',
        'deployment': AZURE_OPENAI_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')


===== APIM Primary Prompt 1: What is the capital of France? =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
The capital of France is **Paris**.

===== APIM Primary Prompt 2: Explain machine learning in one sentence. =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
Machine learning is a field of artificial intelligence where computers learn patterns from data to make predictions or decisions without being explicitly programmed.

===== APIM Primary Prompt 3: Name three primary colors. =====
Status: success
Deployment: chat4o
Model: gpt-4o-2024-11-20
Output:
The three primary colors are red, blue, and yellow.


## Test APIM Gateway Secondary

In [11]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== APIM Secondary Prompt {prompt_index}: {prompt} =====')

    url = f"{APIM_ENDPOINT}{APIM_SECONDARY_PATH}"
    headers = {
        'Ocp-Apim-Subscription-Key': APIM_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'apim-secondary',
        'label': 'apim:gateway-to-secondary',
        'deployment': AZURE_OPENAI_SECONDARY_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')


===== APIM Secondary Prompt 1: What is the capital of France? =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.",
    "type": "invalid_request_error",
    "param": "max_tokens",
    "code": "unsupported_parameter"
  }
}

===== APIM Secondary Prompt 2: Explain machine learning in one sentence. =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.",
    "type": "invalid_request_error",
    "param": "max_tokens",
    "code": "unsupported_parameter"
  }
}

===== APIM Secondary Prompt 3: Name three primary colors. =====
Status: error_400
Deployment: chat51
Model: [not returned]
Output:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this

## Save Combined Results

In [12]:
results_path = Path('../outputs/test-results.json')
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(results, indent=2), encoding='utf-8')

success_count = sum(1 for r in results if r['status'] == 'success')
print(f'\nCompleted checks: {success_count}/{len(results)} succeeded')
print(f'Results file: {results_path}')


Completed checks: 6/12 succeeded
Results file: ..\outputs\test-results.json
